In [1]:
%pip install pylidc pandas

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 12.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 24.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.9/586.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 25.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 26.6 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: Pillow 9.5.0
    Uninstalling Pillow-9.5.0:
      Successfully uninstalled Pillow-9.5.0

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip in

In [2]:
dataset_path = "../dataset/manifest-1762095273065"

In [3]:
!python --version

Python 3.10.12


In [4]:
!echo "" > ~/.pylidcrc

pylidc_config_path = "/root/.pylidcrc"
pylidc_config_content = """
[dicom]
path = /workspace/dataset/manifest-1762095273065/LIDC-IDRI/
warn = True
"""

with open(pylidc_config_path, "w") as f:
    f.writelines(pylidc_config_content)

!cat ~/.pylidcrc


[dicom]
path = /workspace/dataset/manifest-1762095273065/LIDC-IDRI/
warn = True


In [5]:
!grep -rl 'np.int' "/usr/local/lib/python3.10/dist-packages/pylidc" | xargs sed -i 's/\bnp.int\b/int/g'
!grep -rl 'np.float' "/usr/local/lib/python3.10/dist-packages/pylidc" | xargs sed -i 's/\bnp.float\b/float/g'
!grep -rl 'np.bool' "/usr/local/lib/python3.10/dist-packages/pylidc" | xargs sed -i 's/\bnp.bool\b/bool/g'

# Solve pylidc numpy conflict
%pip uninstall -y numpy
%pip install "numpy"
%reset -f

Found existing installation: numpy 1.24.4
Uninstalling numpy-1.24.4:
  Successfully uninstalled numpy-1.24.4
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 23.9 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 24.4.0 requires numpy<2.0a0,>=1.23, but you have numpy 2.2.6 which is incompatible.
cugraph 24.4.0 requires numpy<2.0a0,>=1.23, but you have numpy 2.2.6 which is incompatible.
cugraph-dgl 24.4.0 requires numpy<2.0a0,>=1.23, but you have numpy 2.2.6 which is incompatible.
cugraph-pyg 24.4.0 requires numpy<2.0a0,>=1.23, but you have numpy 2.2.6 which is incompatible.
cugraph-service-server 24.4

In [6]:
import pylidc as pl
import numpy as np
from scipy.ndimage import zoom
import pickle
import os
from tqdm import tqdm

In [7]:
class LIDCDatasetPreprocessor:
    
    def __init__(self, output_dir='lidc_processed', image_size=(512, 512), 
                 voting_threshold=0.5, min_annotations=1):
        self.output_dir = output_dir
        self.image_size = image_size
        self.voting_threshold = voting_threshold
        self.min_annotations = min_annotations
        
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
        os.makedirs(os.path.join(output_dir, 'masks'), exist_ok=True)
    
    def create_voting_mask_2d(self, annotations, slice_idx, volume_shape):
        vote_count = np.zeros((volume_shape[0], volume_shape[1]), dtype=np.float32)
        
        for ann in annotations:
            mask_3d = ann.boolean_mask()
            bbox = ann.bbox()
            
            z_in_mask = slice_idx - bbox[2].start
            
            if 0 <= z_in_mask < mask_3d.shape[2]:
                mask_2d = mask_3d[:, :, z_in_mask]
                y_start, y_end = bbox[0].start, bbox[0].stop
                x_start, x_end = bbox[1].start, bbox[1].stop
                vote_count[y_start:y_end, x_start:x_end] += mask_2d
        
        vote_ratio = vote_count / len(annotations)
        voting_mask = (vote_ratio >= self.voting_threshold).astype(np.uint8)
        
        return voting_mask
    
    def normalize_hu(self, image):
        MIN_BOUND = -1000.0
        MAX_BOUND = 400.0
        
        image = np.clip(image, MIN_BOUND, MAX_BOUND)
        image = (image - MIN_BOUND) / (MAX_BOUND - MIN_BOUND)
        return image.astype(np.float32)
    
    def resize_image(self, image, mask):
        if image.shape == self.image_size:
            return image, mask
        
        zoom_y = self.image_size[0] / image.shape[0]
        zoom_x = self.image_size[1] / image.shape[1]
        
        img_resized = zoom(image, (zoom_y, zoom_x), order=1)
        mask_resized = zoom(mask, (zoom_y, zoom_x), order=0)
        
        return img_resized, mask_resized
    
    def process_scan(self, scan, scan_idx):
        try:
            # --- CRITERION 1: Slice Thickness <= 2.5 mm ---
            # We skip the entire scan if it is too thick (low resolution)
            if scan.slice_thickness > 2.5:
                return 0

            volume = scan.to_volume()
            nodules = scan.cluster_annotations()
            
            if not nodules:
                return 0
            
            # --- CRITERIA 2 & 3: Consensus & Size ---
            valid_nodules = []
            for cluster in nodules:
                # Criterion 2: At least 3 radiologists must agree
                if len(cluster) < 3:
                    continue
                
                # Criterion 3: Average diameter must be >= 3.0 mm
                # We average the diameter estimates from all radiologists in the cluster
                avg_diameter = np.mean([ann.diameter for ann in cluster])
                if avg_diameter < 3.0:
                    continue
                
                valid_nodules.append(cluster)

            if not valid_nodules:
                return 0
            
            # --- EXISTING LOGIC BELOW (Unchanged) ---
            slice_has_nodule = set()
            for nodule in valid_nodules:
                for ann in nodule:
                    for contour in ann.contours:
                        slice_has_nodule.add(int(contour.image_k_position))
            
            processed_count = 0
            
            for slice_idx in sorted(slice_has_nodule):
                if slice_idx >= volume.shape[2]:
                    continue
                
                ct_slice = volume[:, :, slice_idx]
                mask_slice = np.zeros((volume.shape[0], volume.shape[1]), dtype=np.uint8)
                
                for nodule in valid_nodules:
                    nodule_mask = self.create_voting_mask_2d(nodule, slice_idx, volume.shape)
                    mask_slice = np.maximum(mask_slice, nodule_mask)
                
                if mask_slice.max() == 0:
                    continue
                
                ct_normalized = self.normalize_hu(ct_slice)
                ct_resized, mask_resized = self.resize_image(ct_normalized, mask_slice)
                
                sample_id = f"scan_{scan_idx:04d}_slice_{slice_idx:04d}"
                
                np.save(
                    os.path.join(self.output_dir, 'images', f'{sample_id}.npy'),
                    ct_resized
                )
                np.save(
                    os.path.join(self.output_dir, 'masks', f'{sample_id}.npy'),
                    mask_resized
                )
                
                processed_count += 1
            
            return processed_count
            
        except Exception as e:
            print(f"Error processing scan {scan_idx}: {e}")
            return 0
    
    def process_all(self, max_scans=None):
        scans = pl.query(pl.Scan).all()
        
        if max_scans:
            scans = scans[:max_scans]
        
        print(f"Processing {len(scans)} scans...")
        
        total_slices = 0
        metadata = []
        
        for i, scan in enumerate(tqdm(scans)):
            count = self.process_scan(scan, i)
            total_slices += count
            
            if count > 0:
                metadata.append({
                    'scan_idx': i,
                    'patient_id': scan.patient_id,
                    'slice_count': count
                })
        
        with open(os.path.join(self.output_dir, 'metadata.pkl'), 'wb') as f:
            pickle.dump(metadata, f)
        
        print(f"\nProcessing complete!")
        print(f"Total slices extracted: {total_slices}")
        print(f"Data saved to: {self.output_dir}")
        
        return metadata



In [8]:
class LIDCSliceDataset:
    
    def __init__(self, data_dir='lidc_processed', transform=None):
        self.data_dir = data_dir
        self.transform = transform
        
        self.image_dir = os.path.join(data_dir, 'images')
        self.mask_dir = os.path.join(data_dir, 'masks')
       
        self.sample_ids = [
            f.replace('.npy', '') 
            for f in os.listdir(self.image_dir) 
            if f.endswith('.npy')
        ]
        
        print(f"Found {len(self.sample_ids)} slices")
    
    def __len__(self):
        return len(self.sample_ids)
    
    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        
        image = np.load(os.path.join(self.image_dir, f'{sample_id}.npy'))
        mask = np.load(os.path.join(self.mask_dir, f'{sample_id}.npy'))
        
        if len(image.shape) == 2:
            image = image[..., np.newaxis]
        
        if len(mask.shape) == 2:
            mask = mask[..., np.newaxis]
        
        if self.transform:
            image, mask = self.transform(image, mask)
        
        return {
            'image': image,
            'mask': mask,
            'id': sample_id
        }

In [9]:
preprocessor = LIDCDatasetPreprocessor(
    output_dir='luna_lidc_processed',
    image_size=(512, 512),
    voting_threshold=0.5,
    min_annotations=1
)

metadata = preprocessor.process_all()

dataset = LIDCSliceDataset('luna_lidc_processed')

print(f"\nDataset ready with {len(dataset)} slices")

sample = dataset[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Mask shape: {sample['mask'].shape}")
print(f"Sample ID: {sample['id']}")

Processing 1018 scans...


  0%|          | 0/1018 [00:00<?, ?it/s]

Loading dicom files ... This may take a moment.


  0%|          | 2/1018 [00:04<39:05,  2.31s/it]

Loading dicom files ... This may take a moment.


...

Error processing scan 846: Couldn't find DICOM files for Scan(id=847,patient_id=LIDC-IDRI-0808) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0808
Loading dicom files ... This may take a moment.
Error processing scan 847: Couldn't find DICOM files for Scan(id=848,patient_id=LIDC-IDRI-0807) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0807
Loading dicom files ... This may take a moment.
Error processing scan 848: Couldn't find DICOM files for Scan(id=849,patient_id=LIDC-IDRI-0806) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0806
Loading dicom files ... This may take a moment.
Error processing scan 849: Couldn't find DICOM files for Scan(id=850,patient_id=LIDC-IDRI-0805) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0805
Loading dicom files ... This may take a moment.
Error processing scan 850: Couldn't find DICOM files for Scan(id=851,patient_id=LIDC-IDRI-0804) in /workspace/dataset/manifest-1762095273065/LID

 95%|█████████▍| 966/1018 [14:54<00:00, 329.72it/s]

Error processing scan 911: Couldn't find DICOM files for Scan(id=912,patient_id=LIDC-IDRI-0743) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0743
Loading dicom files ... This may take a moment.
Error processing scan 912: Couldn't find DICOM files for Scan(id=913,patient_id=LIDC-IDRI-0742) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0742
Loading dicom files ... This may take a moment.
Error processing scan 914: Couldn't find DICOM files for Scan(id=915,patient_id=LIDC-IDRI-0740) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0740
Loading dicom files ... This may take a moment.
Error processing scan 915: Couldn't find DICOM files for Scan(id=916,patient_id=LIDC-IDRI-0739) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0739
Loading dicom files ... This may take a moment.
Error processing scan 916: Couldn't find DICOM files for Scan(id=917,patient_id=LIDC-IDRI-0738) in /workspace/dataset/manifest-1762095273065/LID

100%|██████████| 1018/1018 [14:54<00:00,  1.14it/s] 

Error processing scan 981: Couldn't find DICOM files for Scan(id=982,patient_id=LIDC-IDRI-0673) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0673
Loading dicom files ... This may take a moment.
Error processing scan 982: Couldn't find DICOM files for Scan(id=983,patient_id=LIDC-IDRI-0672) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0672
Loading dicom files ... This may take a moment.
Error processing scan 983: Couldn't find DICOM files for Scan(id=984,patient_id=LIDC-IDRI-0671) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0671
Loading dicom files ... This may take a moment.
Error processing scan 984: Couldn't find DICOM files for Scan(id=985,patient_id=LIDC-IDRI-0670) in /workspace/dataset/manifest-1762095273065/LIDC-IDRI/LIDC-IDRI-0670
Loading dicom files ... This may take a moment.
Error processing scan 985: Couldn't find DICOM files for Scan(id=986,patient_id=LIDC-IDRI-0669) in /workspace/dataset/manifest-1762095273065/LID